# Tarea 2 Parte A
Integrantes:
- Carlos Raúl Sánchez Figueroa
- Ulises Omar Montes Correa
- Diego Córdoba Gómez

#Inciso 1)

In [0]:
import pyspark.sql.functions as F

In [0]:
df_silver = spark.table("dev.ciencias_data.silver_sessions")
display(df_silver.limit(5))

##Tamaño y estructura de los datos

In [0]:
n_rows = df_silver.count()
n_cols = len(df_silver.columns)

print(f"Número de registros: {n_rows}")
print(f"Número de columnas: {n_cols}")

df_silver.printSchema()

###Interpretación

La tabla silver contiene 29,556 registros con múltiples variables que describen el tráfico de red.
Los tipos de datos son consistentes con la naturaleza de cada variable, permitiendo su análisis posterior.

##Duplicados

In [0]:
total = df_silver.count()
sin_dup = df_silver.dropDuplicates().count()

duplicados = total - sin_dup

print(f"Duplicados: {duplicados}")

In [0]:
total = df_silver.count()
sin_dup = df_silver.select("community_id").dropDuplicates().count()

duplicados = total - sin_dup

print(f"Duplicados: {duplicados}")


###Interpretación

No se identificaron registros completos duplicados en el dataset, aunque si utlizamos el registro unico community id podemos encontrar 3677 registros duplicados. Para no complicarnos de más y observar un por qué, hemos decidido eliminarlos porteriomente al analisis.

##Análisis de valores nulos

In [0]:
df_nulls = df_silver.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_silver.columns
])

df_nulls_long = df_nulls.select(
    F.explode(
        F.array([
            F.struct(F.lit(c).alias("columna"), F.col(c).alias("nulos"))
            for c in df_nulls.columns
        ])
    ).alias("tmp")
).select("tmp.*")

display(df_nulls_long)

Databricks visualization. Run in Databricks to view.

###Interpretación

Se identificó una alta proporción de valores nulos en variables como src_asn, dst_asn, src_geo, dst_geo e init_rtt, y otras variables como community_id.
Esto indica que la información de geolocalización, sistema autónomo y latencia no está disponible para una gran parte de las sesiones.
Por lo tanto, estas variables no se consideran críticas para el análisis posterior.

##Análisis de valores cero

In [0]:
df_zeros = df_silver.select(
    F.count(F.when(F.col("tot_bytes") == 0, True)).alias("tot_bytes"),
    F.count(F.when(F.col("tot_packets") == 0, True)).alias("tot_packets"),
    F.count(F.when(F.col("tot_data_bytes") == 0, True)).alias("tot_data_bytes")
)

df_zeros_long = df_zeros.select(
    F.explode(
        F.array([
            F.struct(F.lit("tot_bytes").alias("variable"), F.col("tot_bytes").alias("zeros")),
            F.struct(F.lit("tot_packets").alias("variable"), F.col("tot_packets").alias("zeros")),
            F.struct(F.lit("tot_data_bytes").alias("variable"), F.col("tot_data_bytes").alias("zeros"))
        ])
    ).alias("tmp")
).select("tmp.*")

display(df_zeros_long)

Databricks visualization. Run in Databricks to view.

###Interpretación

Se observaron 5430 registros con valor cero en tot_data_bytes, lo cual puede representar sesiones sin transferencia efectiva de datos.
Este comportamiento es consistente con tráfico de red y no se considera un error.

##Validación de consistencia

In [0]:
df_consistency = df_silver.withColumn(
    "diff_bytes",
    F.col("tot_bytes") - (F.col("src_bytes") + F.col("dst_bytes"))
)

display(df_consistency.select("diff_bytes").limit(10))
print('¿Cuántos hay distintos de cero?:', df_consistency.select("diff_bytes").filter(F.col("diff_bytes") != 0).count())

Databricks visualization. Run in Databricks to view.

###Interpretación

La diferencia entre tot_bytes y la suma de src_bytes y dst_bytes es igual a cero en todos los registros, lo cual confirma que los datos son completamente consistentes.

##Estadísticas descriptivas

In [0]:
# Veamos el numero de categorias unicas que hay en nuestras variables cateogricas de interes
df_silver = spark.table("dev.ciencias_data.silver_sessions")
df_silver.select(
    F.countDistinct("dst_ip").alias("unique_dst_ip"),
    F.countDistinct("src_ip").alias("unique_src_ip"),
    F.countDistinct("protocol").alias("unique_protocol")
).show()

df_silver.groupBy(F.col("protocol")[0]).count().orderBy("count", ascending=False).show()
df_silver.groupBy("protocol").count().orderBy("count", ascending=False).show()
df_silver.groupBy("dst_ip").count().orderBy("count", ascending=False).show()
df_silver.groupBy("src_ip").count().orderBy("count", ascending=False).show()


### Interpretación
Observemos que tenemos un gran número de categorias en las ip's, aunque en relación a los registros totales, el número de categorias es inferior. Pero notemos que hay una mala representación dentro de las categorias de las ip's siendo demasiado dispersona y no equitativa, mientras que tenemos conteos superiores a 3000 y 5000 registros respectivamente, estos van cayendo abruptamente. En protocolo tenemos un mismo caso de sobre representación pero son menos categorias 

In [0]:
cols = ["tot_bytes", "tot_packets", "tot_data_bytes"]
stats = (
    df_silver
    .select(
        *[F.col(c) for c in cols]
    )
    .summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max")
)

display(stats)


Databricks visualization. Run in Databricks to view.

###Interpretación

Se observa una alta dispersión en las variables numéricas, especialmente en tot_bytes, donde existen valores extremos significativamente mayores al promedio.
Esto indica la presencia de sesiones de alto volumen, lo cual es esperado en tráfico de red.

##Conclusión general
Se realizó un análisis de calidad de los datos en la capa silver, evaluando duplicados, valores nulos, valores cero, consistencia y estadísticas descriptivas.
No se encontraron registros duplicados, lo cual garantiza la integridad del dataset.
Se identificó una alta proporción de valores nulos en variables relacionadas con geolocalización y sistema autónomo, por lo que su uso en el análisis será limitado.
Las variables principales de tráfico (tot_bytes, tot_packets, tot_data_bytes) presentan buena calidad y consistencia.
Se detectaron valores cero en tot_data_bytes, los cuales son coherentes con la naturaleza del tráfico de red.
Asimismo, se observó una alta dispersión en las variables numéricas, con presencia de valores extremos esperados en este tipo de datos.
Finalmente, se confirmó la consistencia total de los datos, lo que permite continuar con confianza hacia la etapa de Feature Engineering y modelado. Para las variables categoricas hemos decidido mantener unicamente protocolo como variable categorica no ordinal. Que a pesar de que nuestras 3 variables categoricas presentaan el mismo problema, al ser demasiadas categorias en las ip's puede sesgar y sobre ajustar nuestros modelos, por lo que hemos decidido unicamente mantener protocol. 

#Inciso 2)


In [0]:
import pyspark.sql.functions as F

df_silver = spark.table("dev.ciencias_data.silver_sessions")

In [0]:
print(df_silver.count())
df_features = df_silver

In [0]:
df_features = (
    df_features
    .withColumn(
        "session_duration",
        (F.col("last_packet").cast("long") - F.col("first_packet").cast("long")).cast("long")
    )
    .withColumn(
        "bytes_per_second",
        F.when(F.col("session_duration") > 0,
               (F.col("tot_bytes") / F.col("session_duration")).cast("double"))
         .otherwise(F.lit(0.0))
    )
    .withColumn(
        "avg_packet_size",
        F.when(F.col("tot_packets") > 0,
               (F.col("tot_bytes") / F.col("tot_packets")).cast("double"))
         .otherwise(F.lit(0.0))
    )
    .withColumn(
        "bytes_ratio_src_dst",
        F.when(F.col("dst_bytes") > 0,
               (F.col("src_bytes") / F.col("dst_bytes")).cast("double"))
         .otherwise(F.lit(0.0))
    )
    .withColumn(
        "packets_ratio_src_dst",
        F.when(F.col("dst_packets") > 0,
               (F.col("src_packets") / F.col("dst_packets")).cast("double"))
         .otherwise(F.lit(0.0))
    )
)



In [0]:
display(df_features.limit(5))

Hemos decidido conservar los registros en donde hay nulos en community ID y optado por un nuevo id usando uuid

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import uuid
def get_uuid():
    return str(uuid.uuid4())
uuid_v4=udf(get_uuid,StringType())
df_features=df_features.withColumn("id",uuid_v4())

In [0]:
df_features = df_features.select(
    "id",
    "protocol",
    "tot_bytes",
    "tot_packets",
    "tot_data_bytes",
    "session_duration",
    "bytes_per_second",
    "avg_packet_size",
    "bytes_ratio_src_dst",
    "packets_ratio_src_dst"
)

In [0]:
df_features.limit(10).display()

In [0]:
df_features.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df_features.columns]).display()

In [0]:
df_features = df_features.dropna()
# Tamaño del dataset
df_features.count()

In [0]:
%sql
use catalog dev;
create schema if not exists feature_store;

In [0]:
%pip install databricks-feature-engineering

In [0]:
%sql
DROP TABLE IF EXISTS dev.ciencias_data.traffic_network_fs

In [0]:
from databricks.feature_engineering  import FeatureEngineeringClient
fe=FeatureEngineeringClient()
fe.create_table(
    name="dev.ciencias_data.traffic_network_fs",
    primary_keys=["id"],
    df=df_features,
    schema=df_features.schema,
    description="features de sesiones de tráfico de red"
)

In [0]:
display(spark.table("dev.ciencias_data.traffic_network_fs").limit(10))
spark.table("dev.ciencias_data.traffic_network_fs").count()

# Inciso 3
###Feature Engineering

Se construyeron nuevas variables a partir de los datos originales con el objetivo de capturar mejor el comportamiento del tráfico de red.
Entre las principales variables generadas se encuentran:

bytes_per_second: mide la velocidad de transmisión de datos por sesión
avg_packet_size: representa el tamaño promedio de los paquetes
bytes_ratio_src_dst: indica la relación entre bytes enviados y recibidos
packets_ratio_src_dst: mide la proporción de paquetes entre origen y destino

Estas variables permiten caracterizar de manera más precisa cada sesión, facilitando la identificación de patrones y anomalías en el tráfico de red.

Finalmente, las features fueron almacenadas en una tabla Delta dentro de Unity Catalog, funcionando como un repositorio centralizado de variables reutilizables para modelos de machine learning.

Ahora haremos un pipeline transformando variables, normalizaremos los datos numericos y haremos onehotencoder para aquellas categorias sin orden

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.feature_engineering import FeatureLookup
from databricks.feature_engineering import FeatureEngineeringClient
from pyspark.ml.feature import VectorAssembler, MinMaxScaler, StringIndexer, CountVectorizer, OneHotEncoder
import pyspark.sql.functions as F
from pyspark.ml import Pipeline


In [0]:
df = spark.table("dev.ciencias_data.traffic_network_fs").select("id")
fe = FeatureEngineeringClient()

def load_data(data, table_name, lookup_key):
    model_feature_lookups = [FeatureLookup(table_name=table_name, lookup_key=lookup_key)]
    training_set = fe.create_training_set(
        df=data,
        feature_lookups=model_feature_lookups,
        label=None
    )
    training_df = training_set.load_df()
    return training_df

# Crear el df_train
df_train = load_data(df, "dev.ciencias_data.traffic_network_fs", "id")
df_train.count()

In [0]:
num_features = [
    'tot_bytes',
    'tot_packets',
    'tot_data_bytes',
    'session_duration',
    'bytes_per_second',
    'avg_packet_size',
    'bytes_ratio_src_dst',
    'packets_ratio_src_dst'
]

vectorizer = CountVectorizer(
    inputCol="protocol", 
    outputCol="protocols_vec", 
    binary=True
)

# string_idx = StringIndexer(
#     inputCols=["src_ip", "dst_ip"],
#     outputCols=["idx_src_ip", "idx_dst_ip"],
#     handleInvalid="skip",
#     stringOrderType="frequencyDesc" 
# )

# ohe = OneHotEncoder(
#     inputCols=["idx_src_ip", "idx_dst_ip"],
#     outputCols=["ohe_src_ip", "ohe_dst_ip"],
#     dropLast=False
# )

# assembler = VectorAssembler(
#     inputCols=["protocols_vec", "ohe_src_ip", "ohe_dst_ip"] + num_features, 
#     outputCol="features_unscaled",
#     handleInvalid="skip" 
# )

assembler = VectorAssembler(
    inputCols=["protocols_vec"] + num_features, 
    outputCol="features_unscaled",
    handleInvalid="skip" 
)

scaler = MinMaxScaler(
    inputCol="features_unscaled", 
    outputCol="features"
)


## K Means

In [0]:
from pyspark.ml.clustering import KMeans
kmeans = KMeans(k=4, seed=42, featuresCol="features", predictionCol="cluster")

pipeline_completo = Pipeline(stages=[vectorizer, assembler, scaler, kmeans])

pipeline_model = pipeline_completo.fit(df_train)
predictions = pipeline_model.transform(df_train)

predictions.display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType
import numpy as np

kmeans_model = pipeline_model.stages[-1]
centers = kmeans_model.clusterCenters()

def get_distance(features, cluster_idx):
    center = centers[cluster_idx]
    return float(np.linalg.norm(features.toArray() - center))

distance_udf = F.udf(get_distance, FloatType())

df_dist = predictions.withColumn(
    "dist_to_centroid", 
    distance_udf(F.col("features"), F.col("cluster"))
)

threshold = df_dist.approxQuantile("dist_to_centroid", [0.98], 0.01)[0]
print(f"Umbral de anomalía detectado: {threshold}")

df_final = df_dist.withColumn("is_anomaly", F.col("dist_to_centroid") > threshold)

df_final.select(
    "protocol", 
    "tot_bytes",
    "tot_packets",
    "tot_data_bytes",
    "session_duration", 
    "cluster", 
    "dist_to_centroid", 
    "is_anomaly"
).show(100, truncate=False)

In [0]:
import matplotlib.pyplot as plt
from pyspark.ml.evaluation import ClusteringEvaluator
import gc 

pipeline_fe = Pipeline(stages=[vectorizer, assembler, scaler])
pipeline_model_fe = pipeline_fe.fit(df_train)
df_features_only = pipeline_model_fe.transform(df_train)


cost = []
k_range = range(2, 10)

print("Calculando Método del Codo...")
for k in k_range:
    km = KMeans(k=k, seed=42, featuresCol="features")
    
    # Entrenar el modelo
    model_km = km.fit(df_features_only) 
    
    # Obtenemos la inercia (costo)
    wcss = model_km.summary.trainingCost
    cost.append(wcss)
    print(f"Para K={k}, el costo WCSS es: {wcss}")
    
    # --- SOLUCIÓN AL ERROR DE MEMORIA ---
    # Eliminamos el modelo de la sesión de Spark Connect
    del model_km
    # Forzamos a Python a limpiar la memoria RAM inmediatamente
    gc.collect() 
    # ------------------------------------

# Graficar el Codo
plt.figure(figsize=(10, 6))
plt.plot(k_range, cost, 'bx-')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Costo (WCSS)')
plt.title('Método del Codo')
plt.show()

El metodo del codo nos indica que el parametro k = 4 inicial que tenemos es el optimo para dividir nuestro datatrain. Podemos concluir esto ya que entre 3 y 4 hay una caida abrupta en la variable costo del algoritmo, formando visualmente un "codo". Ignoramos k=6 ya que creemos que esa caida se debe a un overfitting

In [0]:
from pyspark.sql import functions as F

# Filtramos las anomalías para que no sesguen el perfil normal de los clusters
df_normal = df_final.filter(F.col("is_anomaly") == False)

# Calculamos estadísticas clave por cluster
perfil_clusters = df_normal.groupBy("cluster").agg(
    F.count("*").alias("cantidad_sesiones"),
    F.round(F.avg("tot_bytes"), 2).alias("promedio_bytes"),
    F.round(F.avg("tot_packets"), 2).alias("promedio_paquetes"),
    F.round(F.avg("session_duration"), 2).alias("duracion_promedio_seg"),
    F.round(F.avg("bytes_per_second"), 2).alias("bytes_por_segundo_promedio")
).orderBy("cluster")

perfil_clusters.display()

In [0]:
# Ver la distribución de protocolos dentro de cada cluster
composicion_protocolos = df_normal.groupBy("cluster", "protocol").agg(
    F.count("*").alias("conteo")
).orderBy("cluster", F.desc("conteo"))

composicion_protocolos.display()

In [0]:
# Filtramos solo el tráfico clasificado como anomalía
df_anomalias = df_final.filter(F.col("is_anomaly") == True)

# Identificamos las IPs de origen que más generan anomalías
top_ips_anomalas = df_anomalias.groupBy("protocol").agg(
    F.count("*").alias("intentos_anomalos"),
    F.avg("tot_bytes").alias("bytes_promedio_en_anomalia")
).orderBy(F.desc("intentos_anomalos"))

print("Top IPs generando tráfico anómalo:")
top_ips_anomalas.show(20, truncate=False)

In [0]:
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(
    predictionCol="cluster", 
    featuresCol="features", 
    metricName="silhouette", 
    distanceMeasure="squaredEuclidean"
)
silhouette = evaluator.evaluate(predictions)
print(f"Silhouette : {silhouette}")

### Evaluación de la Cohesión (Silhouette Score: 0.562)

Al principio se intento no descartar las variables categóricas de IP pero al conservar únicamente el protocolo, se notó una mejora drástica en el índice de Silhouette, pasando de 0.208 a 0.562. Esta subida confirma que la alta cardinalidad de las direcciones IP estaba inyectando demasiado ruido y forzando solapamientos artificiales. Con este nuevo valor, queda claro que la estructura de K-Means (con K=4) ahora sí es más sólida, compacta y tiene fronteras de separación mucho más limpias entre los tipos de tráfico.

### Perfilamiento de Conglomerados (Clusters)

La distribución actual me separa el tráfico en cuatro funciones operativas muy claras:

* **Cluster 0 (Resolución de Nombres - DNS puro):** Es el grupo más homogéneo. El 100% de sus 6,700 sesiones son UDP/DNS. Como era de esperarse para este protocolo, las conexiones son extremadamente ligeras (casi 698 bytes promedio) y prácticamente instantáneas (0.48 segundos). 
* **Cluster 1 (Tráfico Web y Texto Plano):** Agrupa sesiones largas (19.88 segundos promedio) de peso considerable (39,069 bytes). Está dominado por conexiones HTTP y TCP puro. Este perfil encaja con la navegación estándar no cifrada y transferencias de aplicaciones internas.
* **Cluster 2 (Tráfico Cifrado de Alta Velocidad):** Este grupo es exclusivamente tráfico TLS sobre TCP. Es el perfil que más estrés pone en la red, con la mayor velocidad de transferencia (8,406 bytes por segundo) y el paquete de datos promedio más pesado (49,832 bytes). Representa navegación segura pesada o descargas encriptadas.
* **Cluster 3 (Monitoreo, Control y Ruido de Fondo):** Es el bloque con más sesiones (10,103). Funciona como un agrupador de tráfico de gestión de red: SNMP, ICMP (pings), LLMNR, y broadcast UDP. Tienen duraciones muy cortas (2.46 segundos) y un peso moderado-bajo (7,411 bytes), clásico del sondeo y telemetría de dispositivos.

### Análisis de Anomalías Aisladas

Al revisar el tráfico que el modelo dejó fuera de los centroides normales, saltan patrones muy específicos que requieren revisión:

* **Túneles o Exfiltración en ICMP:** Llama mucho la atención ver 334 intentos anómalos usando ICMP con un promedio de 865,873 bytes. Un ping normal mide unos pocos bytes; ver casi un megabyte transferido por ICMP sugiere fuertemente un intento de tunneling o exfiltración de datos disfrazada de control de red.
* **Descargas Atípicas (TCP/UDP/TLS/HTTP):** Los protocolos pesados registran anomalías por volumen bruto. Las anomalías en TCP puro (178), UDP (158), TLS (36) y HTTP (71) están promediando entre 7.3 y 10.4 millones de bytes. Son descargas masivas que rompieron la barrera estadística del percentil de control de su propio cluster.
* **Sondeo Irregular:** Aparecen 92 intentos anómalos combinando UDP, LDAP y SNMP. Aunque el peso es bajo (28,367 bytes promedio), la rareza estadística de este paquete combinado sugiere un posible escaneo de vulnerabilidades en servicios de directorio y gestión.

### Conclusión

La decisión de eliminar las direcciones IP fue un acierto total para el modelo de clustering. En lugar de tratar de agrupar miles de nodos individuales dispersos, el algoritmo logró aislar el comportamiento de la red por su naturaleza de protocolo. Ahora tengo 4 perfiles base estadísticamente limpios: DNS puro, tráfico en texto plano, tráfico cifrado y telemetría. Esto no solo hace que el modelo sea mucho más ligero y lógico de interpretar, sino que permite perfilar las anomalías no por "quién" lo hizo (IP), sino por alteraciones drásticas en la volumetría estándar de "cómo" se usa cada protocolo, lo cual resulta mucho más útil para alertar sobre comportamientos sospechosos en la red.

## DBSCAN

In [0]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType


# Extraemos solo el ID y las features a Pandas para usar Scikit-Learn
print("Convirtiendo datos a Pandas...")
df_pandas = df_features_only.select("id", "features").toPandas()

# Convertimos los vectores dispersos (SparseVectors) de PySpark a un arreglo NumPy
print("Preparando matriz matemática...")
X = np.array(df_pandas["features"].apply(lambda x: x.toArray()).tolist())

# Configuramos DBSCAN
# eps = Distancia máxima (radio) para considerar que dos sesiones son "vecinas"
# min_samples = Cuántas sesiones mínimas se necesitan juntas para formar un cluster
dbscan = DBSCAN(eps=0.5, min_samples=50, n_jobs=-1) 

# Entrenamos y predecimos
print("Entrenando DBSCAN")
clusters_dbscan = dbscan.fit_predict(X)

In [0]:
# Agregamos los resultados al DataFrame de Pandas
df_pandas["cluster_dbscan"] = clusters_dbscan.tolist()

# Regresamos los resultados a PySpark
# Solo necesitamos el ID y el cluster para hacer un JOIN rápido y limpio
df_etiquetas = df_pandas[["id", "cluster_dbscan"]]
spark_dbscan_res = spark.createDataFrame(df_etiquetas)


# Unimos las predicciones con tus datos originales de PySpark
df_final_dbscan = df_features_only.join(spark_dbscan_res, on="id", how="left")

# La Magia de DBSCAN: Todo lo que etiqueta como "-1" es ruido (ANOMALÍA)
df_final_dbscan = df_final_dbscan.withColumn(
    "is_anomaly", 
    F.col("cluster_dbscan") == -1
)

# Mostrar resultados
print("Resultados de DBSCAN:")
df_final_dbscan.select(
    "protocol", 
    "tot_bytes",
    "tot_packets",
    "tot_data_bytes",
    "session_duration",
    "bytes_per_second",
    "cluster_dbscan", 
    "is_anomaly"
).show(100, truncate=False)

In [0]:
# Filtramos las anomalías para que no sesguen el perfil normal de los clusters
df_normal = df_final_dbscan.filter(F.col("is_anomaly") == False)

# Calculamos estadísticas clave por cluster
perfil_clusters = df_final_dbscan.groupBy("cluster_dbscan").agg(
    F.count("*").alias("cantidad_sesiones"),
    F.round(F.avg("tot_bytes"), 2).alias("promedio_bytes"),
    F.round(F.avg("tot_packets"), 2).alias("promedio_paquetes"),
    F.round(F.avg("session_duration"), 2).alias("duracion_promedio_seg"),
    F.round(F.avg("bytes_per_second"), 2).alias("bytes_por_segundo_promedio")
).orderBy("cluster_dbscan")

perfil_clusters.display()

In [0]:
# Ver la distribución de protocolos dentro de cada cluster
composicion_protocolos = df_normal.groupBy("cluster_dbscan", "protocol").agg(
    F.count("*").alias("conteo")
).orderBy("cluster_dbscan", F.desc("conteo"))

composicion_protocolos.display()

In [0]:
df_anomalias = df_final_dbscan.filter(F.col("is_anomaly") == True)

# Identificamos las IPs de origen que más generan anomalías
top_ips_anomalas = df_anomalias.groupBy("protocol").agg(
    F.count("*").alias("intentos_anomalos"),
    F.avg("tot_bytes").alias("bytes_promedio_en_anomalia")
).orderBy(F.desc("intentos_anomalos"))

print("Top Protocolos generando tráfico anómalo:")
top_ips_anomalas.show(20, truncate=False)

### Observaciones del Modelo DBSCAN

A diferencia de la partición generalista del modelo anterior, la configuración basada en densidad de DBSCAN (`eps=0.5, min_samples=50`) generó una topología de red mucho más granular, identificando 20 micro-conglomerados válidos (0 a 19) y un grupo específico de ruido o anomalías (-1). 

**1. Separación Natural por Volumen (El efecto de la Densidad)**
El hallazgo más importante de este modelo es su capacidad para separar versiones "estándar" y "pesadas" de un mismo protocolo en distintos clusters legítimos, ya que ambos comportamientos probaron tener suficiente densidad (más de 50 repeticiones) para no ser considerados simples anomalías aisladas:
* **ICMP (Pings vs. Posible Tunneling):** El modelo dividió el protocolo ICMP en dos comportamientos densos. El **Cluster 3** (2,441 sesiones) representa el ping normal, durando 2.7 segundos y pesando solo 773 bytes. En contraste, el **Cluster 4** (331 sesiones) aisló un bloque denso de ICMP atípico, con una duración de 149 segundos y un peso masivo de 873,301 bytes.
* **TCP/HTTP (Navegación vs. Descargas Masivas):** El tráfico HTTP estándar fue encapsulado en el **Cluster 2** (4,006 sesiones, 43 KB promedio). Sin embargo, DBSCAN encontró suficiente densidad para crear el **Cluster 19** (74 sesiones), aislando transferencias HTTP masivas de 9.1 MB a velocidades altísimas (428,662 bytes/segundo).
* **TCP Puro:** Se observa la misma partición con el tráfico TCP estándar (**Cluster 0**, 2,962 sesiones, 8 KB) frente a transferencias TCP industriales masivas (**Cluster 5**, 196 sesiones, 8.1 MB).

**2. Perfilamiento Estándar Intacto**
El tráfico de control y navegación base mantiene una alta cohesión:
* **Cluster 7:** Dominio total de resolución DNS (6,700 sesiones, 697 bytes, 0.48s).
* **Cluster 8:** Telemetría estándar SNMP (4,141 sesiones, 469 bytes, 0.98s).
* **Cluster 1:** Navegación cifrada TLS/TCP (4,730 sesiones, 42.6 KB, 18.4s).

### Análisis de Anomalías Reales (Cluster -1)

El Cluster -1 aisló 212 sesiones de red. A diferencia de los micro-clusters masivos mencionados antes, el tráfico etiquetado como -1 carece de la densidad estadística necesaria para ser considerado un proceso regular, revelando dos vectores críticos:

* **Sondeo y Protocolos Raros:** Destacan protocolos de administración y gestión operando de forma esporádica y con bajo volumen de datos, lo que sugiere mapeos o intentos de acceso manuales. Es el caso de intentos aislados por UDP/LDAP (42), DHCP atípico (23), TACACS (11), y protocolos heredados o específicos como DCERPC, POP3 y SSH.
* **Descargas Cifradas / HTTP Aisladas:** Aunque DBSCAN agrupó la mayoría de las descargas grandes en clusters válidos, detectó 41 intentos en TLS/TCP y 2 intentos en HTTP que no encajan en ningún patrón de densidad, promediando volúmenes extremos de 9.9 MB y 11.7 MB respectivamente. Además, capturó 28 intentos anómalos usando UDP/Syslog transfiriendo volúmenes irreales para su protocolo (1.1 MB promedio).

### Conclusión

La aplicación de DBSCAN optimizó radicalmente el análisis al mapear el tráfico de red mediante micro-perfiles de densidad. En lugar de obligar a forzar el tráfico pesado en un solo grupo de ruido, el algoritmo reconoció que existen operaciones masivas recurrentes (como túneles ICMP o descargas TCP pesadas) que forman parte de la operatividad del sistema (identificados en los clusters 4, 5, 17 y 19). Al limpiar estos falsos positivos volumétricos de la categoría de anomalías, el modelo logró un Cluster -1 compuesto verdaderamente por ruido estadístico: intentos de escaneo esporádicos en protocolos de administración (LDAP, TACACS, SSH) y extracciones de datos masivas y solitarias que rompen cualquier patrón de repetición en la red.

## Isolation Forest

In [0]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from pyspark.sql import functions as F

df_para_iforest = df_features_only 


df_pandas = df_para_iforest.select("id", "features").toPandas()

# Pasamos los vectores de PySpark a una matriz de NumPy
X = np.array(df_pandas["features"].apply(lambda x: x.toArray()).tolist())

params = {
    "n_estimators": 500, # "Entre más mejor"
    "max_samples": "auto",
    "random_state": 42
}
mod_isolation = IsolationForest(**params, n_jobs=-1)

print("Entrenando Isolation Forest...")
predicciones_if = mod_isolation.fit_predict(X)


df_pandas["pred_iforest"] = predicciones_if.tolist()

# Regresamos todo a PySpark usando el id para el cruce
df_etiquetas_if = df_pandas[["id", "pred_iforest"]]
spark_iforest_res = spark.createDataFrame(df_etiquetas_if)

df_final_iforest = df_para_iforest.join(spark_iforest_res, on="id", how="left")

df_final_iforest = df_final_iforest.withColumn(
    "is_anomaly", 
    F.col("pred_iforest") == -1
)

print("Resultados de Isolation Forest:")
df_final_iforest.select("protocol", 
    "tot_bytes",
    "tot_packets",
    "tot_data_bytes",
    "session_duration",
    "bytes_per_second",
    "pred_iforest", 
    "is_anomaly"
).show(100, truncate=False)

In [0]:
from pyspark.sql import functions as F

# Calculamos estadísticas clave agrupando por la predicción de Isolation Forest (1 y -1)
perfil_iforest = df_final_iforest.groupBy("pred_iforest").agg(
    F.count("*").alias("cantidad_sesiones"),
    F.round(F.avg("tot_bytes"), 2).alias("promedio_bytes"),
    F.round(F.avg("tot_packets"), 2).alias("promedio_paquetes"),
    F.round(F.avg("session_duration"), 2).alias("duracion_promedio_seg"),
    F.round(F.avg("bytes_per_second"), 2).alias("bytes_por_segundo_promedio")
).orderBy(F.desc("cantidad_sesiones"))

print("Perfil General (1 = Normal, -1 = Anomalía):")
perfil_iforest.display()

In [0]:
# Ver la distribución de protocolos dentro de tráfico normal y anómalo
composicion_protocolos_if = df_final_iforest.groupBy("pred_iforest", "protocol").agg(
    F.count("*").alias("conteo")
).orderBy(F.desc("pred_iforest"), F.desc("conteo"))

print("Composición Categórica por Etiqueta:")
composicion_protocolos_if.display()

In [0]:
# Filtramos solo el tráfico clasificado como anomalía (-1)
df_anomalias_if = df_final_iforest.filter(F.col("is_anomaly") == True)

# Identificamos los protocolos que más generan anomalías
top_protocolos_anomalos_if = df_anomalias_if.groupBy("protocol").agg(
    F.count("*").alias("intentos_anomalos"),
    F.avg("tot_bytes").alias("bytes_promedio_en_anomalia")
).orderBy(F.desc("intentos_anomalos"))

print("Top Protocolos generando tráfico anómalo (Isolation Forest):")
top_protocolos_anomalos_if.show(20, truncate=False)

### Perfilamiento General: Tráfico Estándar vs. Anomalías

Al aplicar el algoritmo de Isolation Forest, se obtuvo una clasificación binaria que contrasta la línea base operativa de la red frente a los valores atípicos más extremos. Se observó una diferencia volumétrica y temporal drástica entre ambas categorías:

* **Tráfico Normal (Etiqueta 1):** Constituye la inmensa mayoría de la muestra (28,107 sesiones). Presenta un comportamiento ligero y fluido, con un promedio de 5,515 bytes, 19 paquetes y una duración de 8.27 segundos por sesión. Representa la operatividad cotidiana, esperada y segura de la red.
* **Tráfico Anómalo (Etiqueta -1):** El algoritmo logró aislar 1,440 sesiones (aproximadamente el 4.8% del total) con características radicalmente distintas. Las conexiones catalogadas como anomalías promedian más de 3.14 millones de bytes (3.14 MB), superan los 6,000 paquetes y mantienen conexiones sostenidas por casi 96 segundos.

### Análisis de Tráfico Anómalo por Protocolo

Al desglosar las sesiones expulsadas por el modelo, se identificaron vectores de comportamiento estadísticamente irregulares que requieren atención:

* **Posible Exfiltración o Tunneling en ICMP:** Se registraron 332 sesiones anómalas utilizando el protocolo ICMP con un promedio de 871,033 bytes. Dado que el tráfico ICMP regular (pings) maneja tamaños de paquete minúsculos, la transferencia de casi un megabyte por sesión es un indicador crítico de tráfico encapsulado o exfiltración de datos encubierta.
* **Desviaciones Extremas en Tráfico Pesado:** Protocolos destinados a la transferencia de datos registraron anomalías por volúmenes brutos que rompen la distribución normal. Se identificaron sesiones de TCP puro promediando 7.4 MB y UDP con 6.3 MB, así como descargas irregulares masivas en HTTP (2.6 MB) y TLS (2.2 MB).
* **Sobrecarga en Protocolos de Control y Gestión:** Se detectaron anomalías volumétricas en protocolos diseñados para consultas ligeras. Resaltan 15 sesiones de DNS promediando 140 KB (cuando una consulta DNS estándar pesa menos de 1 KB), además de 115 sesiones combinadas de UDP/LDAP/SNMP promediando 37 KB. Estos registros atípicos sugieren posibles intentos de extracción de directorios, escaneos de vulnerabilidades o el uso de técnicas de amplificación.

### Conclusión

La implementación de Isolation Forest proporcionó un enfoque analítico distinto, evaluando el tráfico no por su similitud con otros grupos, sino por su grado de aislamiento o rareza estadística.

## Conclusión General del Análisis

**1. Cómo se definieron los perfiles y grupos**

No se usaron reglas manuales para crear los perfiles. Se tomaron los datos reales (bytes, paquetes, tiempo y protocolo) y se aplicaron tres modelos matemáticos automáticos:

* **K-Means (Agrupar por parecido):** Creó 4 grupos principales buscando qué conexiones se parecían más entre sí. Así separó el tráfico en: consultas rápidas (DNS), navegación segura (TLS), descargas normales (HTTP/TCP) y alertas de monitoreo (SNMP/ICMP).
* **DBSCAN (Agrupar por concentración):** Creó grupos más pequeños y exactos. Demostró que una transferencia muy pesada no es un problema si ocurre todos los días. Logró separar muy bien un ping normal de un envío masivo de datos.
* **Isolation Forest (Aislar los casos extraños):** Este modelo solo divide el tráfico en dos: normal y anormal. Separó de golpe todas las conexiones que tenían pesos y tiempos que no encajaban para nada con el resto de la red.

**2. Calidad de los resultados: ¿Qué se observa?**

Los resultados se validaron revisando las calificaciones de los modelos y comprobando si tenían sentido en el mundo real. Se observó lo siguiente:

* **Quitar las direcciones IP fue clave:** Al principio, K-Means se confundía porque había miles de direcciones IP diferentes (su calificación de calidad era muy baja, 0.208). Al quitar las IPs y usar solo el protocolo, los grupos quedaron mucho más limpios y claros (la calificación subió a 0.562). Esto prueba que es mejor agrupar el tráfico por "qué hace" en lugar de "quién lo hace".
* **Los tres modelos coinciden:** Aunque los tres programas calculan las cosas de forma muy distinta, todos encontraron exactamente las mismas alertas graves de seguridad.
* **Se detectaron problemas reales:** El descubrimiento más importante en los tres modelos fue ver sesiones de ICMP enviando más de 870 KB. Como un ping normal (ICMP) solo envía unos cuantos bytes, esto es una señal muy clara de que alguien está usando este protocolo de control para sacar datos de la red a escondidas.

**Resumen**

La combinación de los modelos funcionó muy bien. K-Means hizo el mapa general de la red, DBSCAN aportó los detalles más finos, e Isolation Forest confirmó cuáles eran los ataques más graves. Trabajando juntos, logran ignorar el tráfico normal de todos los días y entregan alertas claras y útiles.

# Inciso 4
Vamos a recrear el ejemplo visto en clase del clustering jerarquico pero usando enlace completo, lo único diferente es que entre clusters se va a buscar la distancia máximo para conservarla en el siguiente paso, en lugar del minimo como en el enlace simple, vamos a hacer la talacha a mano usando unicamente pandas y numpy

In [0]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)  # Muestra todas las columnas
pd.set_option('display.width', 1000)        # Amplía el ancho máximo de la consola
pd.set_option('display.max_colwidth', None) # Evita que los nombres largos se trunquen

# Definir municipios y matriz
municipios = ["Milpa Alta", "Tláhuac", "Iztapalapa", "Tlalpan", "Xochimilco", "Coyoacán"]
data = [
    [0.0, 33.2, 31.3, 25.7, 11.4, 39.6],   # Milpa Alta
    [33.2, 0.0, 8.6, 18.5, 15.8, 15.4],    # Tláhuac
    [31.3, 8.6, 0.0, 18.7, 16.0, 15.3],    # Iztapalapa
    [25.7, 18.5, 18.7, 0.0, 9.3, 11.1],    # Tlalpan
    [11.4, 15.8, 16.0, 9.3, 0.0, 15.3],    # Xochimilco
    [39.6, 15.4, 15.3, 11.1, 15.3, 0.0]    # Coyoacán
]

df = pd.DataFrame(data, index=municipios, columns=municipios)
# Reemplazar la diagonal con NaN para que los ceros no interfieran al buscar el mínimo
df = df.mask(np.eye(len(df), dtype=bool))

paso = 1
print("Matriz Original:")
print(df, "\n")

k = 3 #número de clusters 
while len(df) > k:
    # 1. Encontrar la distancia mínima y los nombres de los clusters a unir
    min_valor = df.min().min()
    nodo1, nodo2 = df.stack().idxmin()
    
    nuevo_nodo = f"({nodo1}-{nodo2})"
    
    print(f"--- Paso {paso} ---")
    print(f"Uniendo '{nodo1}' y '{nodo2}' (Distancia: {min_valor})")
    
    # 2. Calcular distancias del nuevo cluster al resto (Enlace Simple = min)
    # Filtramos para no comparar el nuevo nodo consigo mismo
    resto_nodos = [n for n in df.columns if n not in [nodo1, nodo2]]
    
    # Seleccionamos las filas de los dos nodos y sacamos el max por columna ya que eso es lo que nos pide el enlace completo
    distancias_nuevo_nodo = df.loc[[nodo1, nodo2], resto_nodos].max(axis=0) # Si estuvieramos realizando el enlace simple aquí usariamos .min(axis=0)
    
    # 3. Actualizar el DataFrame
    df = df.drop(index=[nodo1, nodo2], columns=[nodo1, nodo2])
    
    # Agregar la nueva fila
    df.loc[nuevo_nodo] = distancias_nuevo_nodo
    # Agregar la nueva columna
    df[nuevo_nodo] = distancias_nuevo_nodo
    
    # Asegurar que la diagonal del nuevo nodo sea NaN
    df.loc[nuevo_nodo, nuevo_nodo] = np.nan
    
    print(df, "\n")
    paso += 1

print("Clustering finalizado.")